<a href="https://colab.research.google.com/github/u9828057/Python_Practice/blob/main/Python_%E6%9A%91%E5%81%87%E7%B7%B4%E7%BF%92_08_09%3A%20%E3%80%90Pandas%E3%80%91Google_Play_Store_%E8%B3%87%E6%96%99%E9%9B%86%EF%BC%8C%E5%AF%A6%E5%8B%99%E6%BC%94%E7%B7%B4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [168]:
#======================================================
# Pandas 資料分析: Google Play Store 資料集，實務演練
# 資料工程核心步驟: 蒐集資料 → 資料清理 → 分析資料
#======================================================
import pandas as pd
print("--------(Pandas: Google Play Store 資料集，實務演練)--------")

# 取得與讀取 CSV 資料

# 【方法一】 雲端環境/公開資料通用法 (例如: google Colab)
# 從 GitHub 上的網址，將 "Google Play Store 資料集" 這個 CSV 檔案的網址存入變數 "csv_url"
csv_url= "https://raw.githubusercontent.com/" \
"u9828057/Python_Practice/refs/heads/main/googleplaystore.csv"

# 透過 Pandas 直接前往該網站讀取資料，並轉換為 DataFrame
df= pd.read_csv(csv_url)

# 【方法二】 本地端開發標準做法 (例如 VS Code)
# 只要檔案與程式碼在同一個資料夾，直接打檔名即可
# df= pd.read_csv("googleplaystore.csv")

#-------------------------------------------------------------

# 觀察資料

print(f"\n{' '*27}【原始的資料形式】\n{df}\n")
print(f"【資料數量】: {df.shape}")
print(f"【資料欄位】: {df.columns}\n")

# 分析資料: 評分的各種統計數據

print(f"【觀察 應用程式的評分 'Rating' 的資料形式】\n{df['Rating']}\n")
print(f"【'Rating' 的平均數】: {df['Rating'].mean():.2f}")
print(f"【'Rating' 的中位數】: {df['Rating'].median():.2f}")
print(f"【取得前一百個應用程式的的平均評分值】: {df['Rating'].nlargest(100).mean():.2f}")
# 發現異常: 觀察 "前一百個應用程式的平均評分值" 時，發現平均值大於 最高評分值 5.0 (Google Play 評分滿分應該是 5.0)

#-------------------------------------------------------------

# 條件篩選，找出異常值

# 目標: 篩選出應用程式評分大於 5.0 的資料
condition_rating= df["Rating"] > 5.0
filtered_data_rating= df[condition_rating]
print(f"\n{' '*20}【觀察 評分值大於 5.0 的應用程式】\n{filtered_data_rating}\n")
# 觀察發現 "10472  Life Made WI-Fi..." 這筆資料整個欄位的資料位移了，且評分 "Rating" 為 19.0

# 條件篩選: 只保留評分小於等於 5.0 的合理資料
condition_rating_lessthan5= df["Rating"] <= 5.0
filtered_data_rating_lessthan5= df[condition_rating_lessthan5]

# 驗證清理結果: 在乾淨的資料中，再次取前一千個最高評分值的資料來計算平均值
print(f"【取得前一千個 '評分值小於等於 5.0' 的應用程式平均評分值】: \
{filtered_data_rating_lessthan5['Rating'].nlargest(1000).mean():.2f}\n")
# 結果: 平均值沒有超過滿分 5.0，代表髒資料已經被成功排除了


#==================================
# 分析資料: 安裝數量的各種統計數據
#==================================
print("--------(分析資料: 安裝數量的各種統計數據)--------")

# 目標: 計算應用程式的平均安裝數
# 難點: 原始資料是字串形式，例如 "10,000+" 或是混入奇怪的字串 "Free"，無法直接做數學運算

# 觀察原始資料
print(f"\n【觀察 應用程式的安裝數量 'Installs' 的資料形式】\n{df['Installs']}\n")

# 取出 Installs 欄位的資料
installs_data= df["Installs"]


#----------------------------
# 除錯過程 (Trial & Error)
#----------------------------

#【嘗試 1】: 直接拔除逗號，並嘗試轉換為數字
# installs_data= pd.to_numeric(installs_data.str.replace(",","", regex=False))
# 發生錯誤: ValueError: Unable to parse string "10000+"
# 失敗原因: 字串裡面還有 "+" 號，無法轉換為數字

#【嘗試 2】: 拔除逗號後，再拔除加號，並嘗試轉換為文字  (這裡使用了連續呼叫 .str.replace() 的技巧)
# installs_data= installs_data.str.replace(",", "", regex=False).str.replace("+", "", regex=False)
# installs_data= pd.to_numeric(installs_data)
# 發生錯誤: ValueError: Unable to parse string "Free" at position 10472
# 失敗原因: 發現了另一個字串異常值 "Free"

# 觀察字串異常值 "Free" 的資料
print(f"【觀察 字串異常值 (索引 10472)】: {installs_data[10472]}\n")


#-----------------------------------------------------------------------------
# 資料清理流程: 排除異常值 → 拔除異常符號 → 轉換為數值 → 將乾淨的資料覆蓋原始 df
#-----------------------------------------------------------------------------

# 步驟 1: 建立條件，排除內容為 "Free" 的異常資料
condition_installs= installs_data != "Free"
clean_installs= installs_data[condition_installs]

# 步驟 2: 拔除逗號與加號 (取代為空字串)
clean_installs= clean_installs.str.replace(",","", regex=False).str.replace("+","", regex=False)

# 步驟 3: 將資料轉換為數值 (pd.to_numeric())
numeric_installs= pd.to_numeric(clean_installs)

# 步驟 4: 把整理乾淨的資料，覆蓋回原本有異常值的 DataFrame 裡
# 將排除掉 "Free" 的乾淨資料，覆蓋回原本的整個 df 裡
df= df[condition_installs]
# 再把整理乾淨的 ["Installs"] 這欄資料覆蓋回 df["Installs"] 裡
df["Installs"]= numeric_installs


#---------------------
# 數據分析與結果驗證
#---------------------

# 成功轉換後，計算平均值
print(f"【'Installs' 的平均數】: {numeric_installs.mean():.2f}")

# 觀察安裝數量大於 100000 的應用程式有幾個
condition_installs_mt100k= df["Installs"] > 100000
filtered_data_installs_mt100k= df[condition_installs_mt100k]
print(f"【觀察 應用程式安裝數量大於 100000 的資料形式】: {filtered_data_installs_mt100k.shape}")
print(f"【觀察 應用程式安裝數量大於 100000 的數量】: {filtered_data_installs_mt100k.shape[0]}\n")


# §補充資料:「為什麼不能直接計算安裝量的平均值？」：
# 在 Python 中，"10" + "20" 結果是字串拼接的 "1020"，而不是數學的 30。
# 這是處理真實資料最常見的錯誤，一定要先拔除符號並轉型 (pd.to_numeric()) 才能算平均值

# §補充資料:「regex=False 是什麼？」：
# 在使用 .str.replace("+", "") 時，會建議加上 regex=False
# 因為 + 在程式裡通常有「正規表達式 (Regular Expression)」的特殊涵義，
# 加上這個參數可以告訴 Pandas：「只是單純想替換字元，不要想太多」，能避免引發警告或預期外的錯誤


#========================================
# 實務應用: 關鍵字搜尋應用程式名稱
#========================================
print("--------(實務應用: 關鍵字搜尋)--------\n")

# 目標: 找出應用程式名稱 (App) 中，包含特定關鍵字的資料
# 難點: 真實情況中，關鍵字可能會有大小寫混雜的情況，需要進行「忽略大小寫」的模糊搜尋

# 1.設定想搜尋的關鍵字
keyword= input("請輸入關鍵字: ")

# 2.建立搜尋條件: 使用 .str.contain() 判斷字串是否包含關鍵字
# 補充技巧: 加上 "case=False" 這個參數，可以忽略大小寫
condition_keyword= df["App"].str.contains(keyword, case=False)

# 3.將條件套用到完整的 DataFrame 上
search_result= df[condition_keyword]

# 4.精緻化輸出結果
# 原始資料有 13 個欄位，全部印出來會很雜亂
# 所以只挑選 ["App", "Rating", "Installs"] 這三個比較重要的欄位來預覽
# §語法: 挑選多個欄位時，需要使用雙層中括號 [[]] 包起來
display_result= search_result[["App", "Rating", "Installs"]]

# 5.觀察最終分析結果
print(f"【搜尋包含關鍵字 '{keyword}' 的應用程式總數】: {search_result.shape[0]} 款\n")
print(f"{' '*18}【搜尋關鍵字 '{keyword}' 結果預覽 (前 30 筆)】\n{display_result.head(30)}\n")












#=============================================================
# §補充資料 1: Google Colab 的儲存空間是暫時的，
# 每次關閉網頁或閒置太久導致「工作階段中斷」後，上傳的 googleplaystore.csv 就會被系統清空
# 下次重新打開這份筆記本練習時，必須再次上傳該檔案，程式才能正常運作
#=============================================================

#=============================================================
# §補充資料 2: 需要使用 with open(...) as file: 來開啟嗎？
# 答案是：不需要，使用 pd.read_csv() 就夠了
#=============================================================

# 這個問題釐清了 Python「原生功能」與「擴充套件」的差別:

# with open(...) as file:
# 這是 Python 內建（原生）的檔案處理方式。當要讀取純文字檔 (.txt)、寫入日誌、或是爬蟲抓取網頁原始碼存檔時，
# 這是標準且安全的做法，因為 with 語法會確保檔案讀寫完畢後自動關閉，不佔用系統資源

# pd.read_csv(...)
# 這是 Pandas 套件 專門為資料分析打造的「超級工具」。
# 當呼叫這行程式碼時，Pandas 在底層其實已經自動做完了 open() 開啟檔案、讀取資料、轉換格式，以及關閉檔案的所有動作

# 結論：
# 當面對結構化的資料表（如 CSV、Excel）並且已經 import pandas 時，
# 直接交給 Pandas 處理即可，不需要把它包在 with open: 裡面，這樣反而會讓程式碼變得複雜
# 業界在做資料分析時，也都是直接使用 pd.read_csv() 到底


--------(Pandas: Google Play Store 資料集，實務演練)--------

                           【原始的資料形式】
                                                     App             Category  \
0         Photo Editor & Candy Camera & Grid & ScrapBook       ART_AND_DESIGN   
1                                    Coloring book moana       ART_AND_DESIGN   
2      U Launcher Lite – FREE Live Cool Themes, Hide ...       ART_AND_DESIGN   
3                                  Sketch - Draw & Paint       ART_AND_DESIGN   
4                  Pixel Draw - Number Art Coloring Book       ART_AND_DESIGN   
...                                                  ...                  ...   
10836                                   Sya9a Maroc - FR               FAMILY   
10837                   Fr. Mike Schmitz Audio Teachings               FAMILY   
10838                             Parkinson Exercices FR              MEDICAL   
10839                      The SCP Foundation DB fr nn5n  BOOKS_AND_REFERENCE   
10840      iHorosc

In [21]:
print(f"Google Play Store 資料集原網址: 'https://www.youtube.com/redirect?event=video_description&redir_token=QUM4Zm9rVDNoZlpudTNwVVdfQmRBUzYtSFZ4RXxBR3JiS2Ftd2Q5REp0RGR4eEN3UEJPd1QxRlRteU1OR3V0cHN6d09FZFV1bUc3Nm00VDFFN2pXaGlvZk1LdzZQN0ZuZzJINXhQVVNoS0pib3Z0WUlmNTdyeFB2Y3JFN2hBc2kx&q=https%3A%2F%2Fcwpeng.github.io%2Flive-records-samples%2Fdata%2Fgoogleplaystore.csv&v=B5BgPWBZhvY'")

Google Play Store 資料集原網址: 'https://www.youtube.com/redirect?event=video_description&redir_token=QUM4Zm9rVDNoZlpudTNwVVdfQmRBUzYtSFZ4RXxBR3JiS2Ftd2Q5REp0RGR4eEN3UEJPd1QxRlRteU1OR3V0cHN6d09FZFV1bUc3Nm00VDFFN2pXaGlvZk1LdzZQN0ZuZzJINXhQVVNoS0pib3Z0WUlmNTdyeFB2Y3JFN2hBc2kx&q=https%3A%2F%2Fcwpeng.github.io%2Flive-records-samples%2Fdata%2Fgoogleplaystore.csv&v=B5BgPWBZhvY'
